# M25 E2E Test — Unified Colab Notebook

Runs the full M25 chain test against a remote ComfyUI instance.

**How to use:**
1. Upload your `agent/` project to Google Drive root (`/content/drive/MyDrive/agent/`)
2. Run cells 1→2→3→4 in order
3. If GPU disconnects — re-run cells 3→4 (Cell 3 auto-detects new URL)

**Requirements:**
- Runtime → Change runtime type → T4 GPU

In [ ]:
#@title ## Cell 1: Mount Google Drive + Link Agent
#@markdown Mounts Drive and links `/content/drive/MyDrive/agent` → `/content/agent`

import os, sys

# Mount Drive
if not os.path.exists('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted')
else:
    print('Drive already mounted')

# Link agent
AGENT_DIR = '/content/drive/MyDrive/agent'
AGENT_LINK = '/content/agent'

if not os.path.exists(AGENT_DIR):
    print(f'ERROR: {AGENT_DIR} not found!')
    print('Upload your agent project to Google Drive root first.')
    print('Expected structure: /content/drive/MyDrive/agent/app/, tests/, workflows/')
else:
    if os.path.islink(AGENT_LINK):
        os.unlink(AGENT_LINK)
    os.symlink(AGENT_DIR, AGENT_LINK)
    sys.path.insert(0, AGENT_LINK)
    
    # Verify structure
    required = ['app', 'tests', 'workflows']
    missing = [d for d in required if not os.path.isdir(os.path.join(AGENT_DIR, d))]
    if missing:
        print(f'WARNING: Missing dirs in agent: {missing}')
        print(f'Contents of {AGENT_DIR}:')
        print(os.listdir(AGENT_DIR))
    else:
        print(f'OK: Agent linked from Drive')
        print(f'  Source: {AGENT_DIR}')
        print(f'  Link:   {AGENT_LINK}')
        print(f'  Files:  {len(os.listdir(AGENT_DIR))} items')

In [ ]:
#@title ## Cell 2: Setup ComfyUI + Install Dependencies
#@markdown Clones ComfyUI, installs requirements, starts server on :8188

import subprocess, time, urllib.request

COMFY_DIR = '/content/ComfyUI'
AGENT_DIR = '/content/drive/MyDrive/agent'

# --- ComfyUI setup ---
if not os.path.exists(COMFY_DIR):
    print('Cloning ComfyUI...')
    !git clone https://github.com/comfyanonymous/ComfyUI $COMFY_DIR 2>&1 | tail -3
    print('Installing ComfyUI requirements...')
    !pip install -r $COMFY_DIR/requirements.txt -q 2>&1 | tail -3
else:
    print('ComfyUI already cloned')

# --- Agent requirements ---
req = os.path.join(AGENT_DIR, 'requirements.txt')
if os.path.exists(req):
    print('Installing agent requirements...')
    !pip install -r $AGENT_DIR/requirements.txt -q 2>&1 | tail -3
else:
    print('No requirements.txt in agent, skipping')

# --- Check if ComfyUI is running ---
def comfy_ready():
    try:
        r = urllib.request.urlopen('http://localhost:8188/system_stats', timeout=5)
        return r.status == 200
    except:
        return False

if comfy_ready():
    print('OK: ComfyUI already running on :8188')
else:
    print('Starting ComfyUI...')
    proc = subprocess.Popen(
        ['python', 'main.py', '--listen', '0.0.0.0', '--port', '8188'],
        cwd=COMFY_DIR,
        stdout=open('/tmp/comfyui.log', 'w'),
        stderr=subprocess.STDOUT
    )
    for i in range(60):
        time.sleep(2)
        if comfy_ready():
            print(f'OK: ComfyUI ready in {i*2}s')
            break
    else:
        print('FAIL: ComfyUI did not start in 120s')
        print('--- Last 500 chars of log ---')
        print(open('/tmp/comfyui.log').read()[-500:])

# --- Quick stats ---
if comfy_ready():
    stats = json.loads(urllib.request.urlopen('http://localhost:8188/system_stats').read())
    device = stats['devices'][0]
    print(f'GPU: {device["name"]}')
    print(f'VRAM: {device["vram_total"]//1024**3}GB total, {device["vram_free"]//1024**3}GB free')
    print(f'ComfyUI: {stats["system"]["comfyui_version"]}')

import json  # ensure json available for stats above

In [ ]:
#@title ## Cell 3: Start Cloudflare Tunnel + Auto-detect URL
#@markdown Starts cloudflared tunnel, parses URL from logs, sets COMFY_REMOTE_URL

import re, subprocess, time, urllib.request, os

LOG = '/tmp/cloudflared.log'

# Kill old cloudflared
!pkill cloudflared 2>/dev/null || true
time.sleep(2)

# Start new tunnel
print('Starting cloudflared tunnel...')
proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8188'],
    stdout=open(LOG, 'w'),
    stderr=subprocess.STDOUT
)

# Wait for URL
url = None
for i in range(30):
    time.sleep(2)
    try:
        text = open(LOG).read()
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', text)
        if match:
            url = match.group(0)
            break
    except:
        pass

if url:
    os.environ['COMFY_REMOTE_URL'] = url
    # Verify tunnel works
    time.sleep(3)
    try:
        r = urllib.request.urlopen(f'{url}/system_stats', timeout=15)
        stats = json.loads(r.read())
        device = stats['devices'][0]
        print(f'OK: Tunnel ready')
        print(f'  URL:   {url}')
        print(f'  GPU:   {device["name"]}')
        print(f'  Setup: COMFY_REMOTE_URL={url}')
    except Exception as e:
        print(f'WARN: URL found but not reachable: {e}')
        print('Wait 10s and re-run this cell')
else:
    print('FAIL: No URL found in cloudflared logs')
    print('Log contents:')
    print(open(LOG).read()[-1000:])

import json

In [ ]:
#@title ## Cell 4: Run M25 E2E Test
#@markdown Runs smoke check + full M25 chain test

import os
url = os.environ.get('COMFY_REMOTE_URL')

if not url:
    print('ERROR: COMFY_REMOTE_URL not set. Re-run Cell 3.')
else:
    print(f'Target: {url}')
    print('Running M25 E2E test...')
    print('=' * 60)
    !cd /content/agent && python tests/_m25_e2e_runner.py --url $COMFY_REMOTE_URL